<h1><center>Language translation (English to Spanish)</center></h1>

In [2]:
# Import libraries
from pathlib import Path
import torch
import torch.nn as nn

from src.data.data_loader import create_dataloaders
from src.model.transformer import build_transformer
from src.model.transformer import Transformer
from src.train.training import train_model
from src.utils.utils import get_device
from nltk.tokenize import word_tokenize
from pathlib import Path

from src.utils.constants import PADDING_ID, UNKNOWN_ID, START_OF_SENTENCE_ID, END_OF_SENTENCE_ID
from src.utils.constants import PADDING_VALUE, UNKNOWN_VALUE, START_OF_SENTENCE_VALUE, END_OF_SENTENCE_VALUE

import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = '0.0'

In [5]:
# Initialize model and training parameters

translation_file = 'data/en-fr.csv'

# Size of embedding vector
d_model = 512
# Number of words in a source vocabulary
src_vocab_size = 14500
# Number of words in a target vocabulary
tgt_vocab_size = 29300
# Max sequence length for input words/tokens
src_seq_len = 75
# Max sequence length for output words/tokens
tgt_seq_len = 75
# Dropout rate
dropout = 0.1
# number of encoder blocks
num_layers = 4
# number of attention heads
num_heads = 8
# Number of hidden nodes for feed-forward layer
d_ff = 4*d_model

# Number of epochs
epochs = 10
# Batch size for training
batch_size = 512

In [6]:
# Get a device to use for training/inference
device = get_device()

# Create training and validation data loaders
train_dataloader, val_dataloader, src_word_to_id, tgt_word_to_id = create_dataloaders(translation_file,
                                                                                      batch_size,
                                                                                      src_seq_len, 
                                                                                      tgt_seq_len, 
                                                                                      src_vocab_size,
                                                                                      tgt_vocab_size)

In [7]:
# Create transformer model
transformer_model = build_transformer(d_model, src_vocab_size, tgt_vocab_size, src_seq_len, tgt_seq_len, 
                                      dropout, num_layers, num_heads, d_ff).to(device)

print(transformer_model)

Transformer(
  (src_embed): Embedding(
    (embedding): Embedding(14500, 512)
  )
  (tgt_embed): Embedding(
    (embedding): Embedding(29300, 512)
  )
  (src_pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (tgt_pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0-3): 4 x EncoderBlock(
        (self_attention): MultiHeadAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (query_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (key_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (value_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (output_linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (feed_forward): FeedForward(
          (linear_1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=Fal

In [8]:
# Create optimizer and loss function
loss_fn = nn.CrossEntropyLoss(ignore_index=PADDING_VALUE)
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

# Train model
train_model(epochs, transformer_model, train_dataloader, val_dataloader,
            loss_fn, optimizer, device)

Train Batch - 0, Loss - 10.302278518676758
Train Batch - 50, Loss - 6.980384826660156
Train Batch - 100, Loss - 5.729327201843262
Train Batch - 150, Loss - 5.309844970703125
Train Batch - 200, Loss - 4.870316505432129
Train Batch - 250, Loss - 4.5809173583984375
Train Batch - 300, Loss - 4.220632553100586
Validation Batch - 0, Loss - 4.169132232666016
Epoch - 1, Train Loss - 5.538175582885742, Validation Loss - 4.144060134887695
Train Batch - 0, Loss - 4.129352569580078
Train Batch - 50, Loss - 3.84417724609375
Train Batch - 100, Loss - 3.8578333854675293
Train Batch - 150, Loss - 3.6433632373809814
Train Batch - 200, Loss - 3.490490436553955
Train Batch - 250, Loss - 3.3652498722076416
Train Batch - 300, Loss - 3.341472625732422
Validation Batch - 0, Loss - 3.261728286743164
Epoch - 2, Train Loss - 3.620969533920288, Validation Loss - 3.213275671005249
Train Batch - 0, Loss - 3.276742935180664
Train Batch - 50, Loss - 3.0666637420654297
Train Batch - 100, Loss - 3.0812973976135254
Tra

In [3]:
# Save model

# Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Create model save path
MODEL_NAME = "08_language_translation.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

In [10]:
# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=transformer_model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models/08_language_translation.pth
